# Exponent- en mantissatruncatie op dezelfde MX-checkpoints

Deze notebook analyseert uitsluitend de opgeslagen **post-hocevaluatie**. De twee vaste checkpoints zijn **epoch 100** en het tijdens de directe gezamenlijke training gekozen **E4M3-bestcheckpoint**. Voor beide gebruiken alle acht routes hetzelfde checkpoint en dezelfde testbeelden.

De directe E4M3-, E2M3- en E2M1-exports vormen de controles. Mantissatruncatie wordt opnieuw getest vanuit E2M3; exponenttruncatie vertrekt vanuit E4M3. De gecombineerde route verwijdert eerst exponentbits en daarna mantissabits. Alle afleidingen behouden hun bronschaal, met A32 en de reeds getrainde **BN-bank van de doelprecisie**. Er is geen training, nieuwe checkpointselectie, BN-herkalibratie of schaalcorrectie.


In [ ]:
from pathlib import Path
RUN_DIR = Path(".")  # Door de evaluator ingevuld; wijs anders naar de nieuwe _evaluate-map.
SAVE_FIGURES = False  # True schrijft een nieuwe analysemap; ruwe bestanden blijven behouden.


In [ ]:
import importlib.util
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

helper_path = (RUN_DIR / "notebook_support.py").resolve()
spec = importlib.util.spec_from_file_location("mx_triple_posthoc_analysis", helper_path)
analysis = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = analysis
spec.loader.exec_module(analysis)
run = analysis.load_run(RUN_DIR, verify_per_image=True)
figures = {}
display(Markdown(analysis.status_markdown(run)))
display(analysis.diagnostics_table(run))


## 1. Acht evaluatieroutes, drie doelprecisies

De routes zijn **geen acht zelfstandig getrainde precisies**. De drie directe controles zijn de tijdens training gebruikte gewichtsmodi. De overige routes vervangen uitsluitend hun MX-gewichtsrepresentatie tijdens inferentie.

Voor mantissatruncatie: `[s ee mmm] → [s ee m]`, dus de twee laagste mantissabits worden verwijderd. Voor exponenttruncatie: `[s e3 e2 e1 e0 mmm] → [s e3 e2 mmm]`; teken en mantissa blijven behouden en de twee hoogste exponentbits worden opnieuw als E2-exponentveld geïnterpreteerd. Dit is geen exponentrebiasing. De gecombineerde route past beide bewerkingen achtereenvolgens toe.

De twee afrondingsroutes zijn controles vanuit opgeslagen bronelementen. **Afronden E4M3 → E2M3 houdt ook de E4M3-bronschaal vast** en kan daarom sterk satureren. Het is geen nieuwe directe MX-kwantisatie met een aangepast schaalbereik.


In [ ]:
display(analysis.route_table())


## 2. Accuracies op epoch 100 en het vaste E4M3-bestcheckpoint

Iedere groep gebruikt één en hetzelfde checkpoint. De drie rechtstreekse controls worden opnieuw op dezelfde beelden gemeten als de truncatieroutes. Een lage truncatieaccuracy is een mogelijke onderzoeksuitkomst en op zichzelf geen technische FAIL.


In [ ]:
display(analysis.accuracy_table(run).style.format({"top1_percent": "{:.4f}%", "hard_cross_entropy": "{:.6f}"}))
figures["checkpoint_accuracy"] = analysis.plot_checkpoint_accuracy(run)
plt.show()


## 3. Verlies of winst ten opzichte van de directe doelmodus

Mantissa 6→4 en combinatie 8→6→4 worden met directe E2M1 vergeleken; exponent 8→6 met directe E2M3. Positief betekent dat de afgeleide route meer correcte voorspellingen geeft. De afrondingscontroles tonen het verschil tussen eerst een opgeslagen bron vormen en daarna afronden of trunceren.

Accuracyverschillen kunnen rekenkundig worden opgeteld, maar vormen geen onafhankelijke causale ontbinding. De ingrepen werken door een niet-lineair netwerk met vaste, eerder op directe gewichten geleerde BN-statistieken.


In [ ]:
display(analysis.delta_table(run).style.format({"Delta accuracy [pp]": "{:+.4f}"}))
figures["accuracy_deltas"] = analysis.plot_accuracy_deltas(run)
plt.show()


## 4. Gepaarde voorspellingen: welke beelden winnen of verliezen?

Een klein netto verschil kan samengaan met veel gewijzigde voorspellingen. De tabel telt daarom fout→correct, correct→fout en fout→andere fout. Bij de directe routes helpt dit ook om de kleine accuracyverschillen tussen precisies te beschrijven. Uit twee checkpoints van één training volgt geen algemene superioriteits- of equivalentieclaim.

De richting is steeds `other_path − reference_path`. De notebook controleert de tellingen opnieuw vanuit de opgeslagen voorspellingen, zonder het model te laden.


In [ ]:
pair_columns = [c for c in ["role", "epoch", "reference_path", "other_path", "accuracy_difference_pp", "prediction_changes", "wrong_to_correct", "correct_to_wrong", "wrong_to_other_wrong", "net_correct_change"] if c in run.pairs]
display(run.pairs[pair_columns].style.format({"accuracy_difference_pp": "{:+.4f}"}))
figures["paired_changes"] = analysis.plot_paired_changes(run)
plt.show()
display(pd.DataFrame(run.output_checks))


## 5. Wat gebeurt er met de gewichten en de schaal?

Deze afwijkingen vergelijken **gedecodeerde gewichten tussen routes**. Zij zijn geen foutmaat tegenover de FP32-mastergewichten; die worden in deze proef niet ingelezen. Padding telt niet mee als echt gewicht.

De eerste figuur toont de MAE tegenover de directe doelmodus. De tweede toont de RMS-grootteverhouding tegenover diezelfde directe modus. Een verhouding ver van 1 maakt een verandering in gewichtsamplitude zichtbaar. De behouden E4M3-bronschaal kan bij een E2-doelformaat een groot effect hebben; er wordt hier niets gecompenseerd of opnieuw geleerd.


In [ ]:
code_columns = [c for c in ["role", "epoch", "layer", "path", "n_weights", "mean_abs", "rms", "rms_ratio_to_direct", "zero_count", "local_subnormal_count", "scale_exp_min", "scale_exp_max", "scale_anchor_path", "scales_equal_source", "nonzero_dropped_exponent_suffix_count", "nonzero_dropped_mantissa_suffix_count"] if c in run.codes]
display(run.codes[code_columns])
display(run.weights)
figures["weight_differences"] = analysis.plot_weight_differences(run)
plt.show()
figures["weight_magnitudes"] = analysis.plot_weight_magnitudes(run)
plt.show()


## 6. Reproductie en uitvoeringscontroles

De opnieuw gemeten directe scores worden vergeleken met hun historische trainingsscores. Een verschil blijft zichtbaar en wordt niet gecorrigeerd om de oude score te reproduceren. De nieuwe, gelijktijdig gemeten vergelijkingen tussen routes zijn leidend voor deze post-hocproef.

De bestaande checkpointkeuze blijft vast. Het E4M3-bestcheckpoint was geselecteerd op de officiële testset; de score daarvan is geen onafhankelijke testschatting na selectie. Alle doel-BN-banken blijven bevroren. Dit is numerieke MX-emulatie met FP32-rekenbuffers, geen meting van native-MX-hardwarewinst.


In [ ]:
display(run.reproduction)
display(analysis.audit_table(run))
display(Markdown(analysis.timing_markdown(run)))
if run.report.get("limitations"):
    display(Markdown("\n".join("- " + str(item) for item in run.report["limitations"])))


## 7. Figuren bewaren

Zet bovenaan `SAVE_FIGURES = True` en voer deze cel uit om PNG- en PDF-figuren in een **nieuwe** analysemap op te slaan. De ruwe rapporten, voorspellingen, exports en eerdere resultaten worden niet overschreven. Deze notebook slaat geen extra uitgevoerde notebookkopie op.


In [ ]:
if SAVE_FIGURES:
    saved = analysis.save_figures(run, figures)
    print("Figuren opgeslagen in:", saved)
else:
    print("Figuuropslag staat uit; de bestaande bestanden zijn alleen gelezen.")
